DIY Disease Tracking Dashboard Kit (C) Fabrizio Smeraldi, 2020,2024 (f.smeraldi@qmul.ac.uk - web). This notebook is released under the GNU GPLv3.0 or later.

Dashboard customisation and additional code (C) Hannah Thirsk, 2025

# Influenza Surveillance Dashboard

## About This Dashboard

This dashboard shows influenza data from the UK Health Security Agency (UKHSA). There are two main things it displays:

1. **ICU/HDU Admission Rates** - how many people get admitted to intensive care with flu (per 100,000 people)
2. **Testing Positivity by Age Group** - what percentage of flu tests are positive, split up by age

The dashboard loads data from files when it starts up, but you can also refresh it to get the latest data.

In [1]:
# importing everything I need
# pandas and matplotlib for the data and graphs, ipywidgets for the buttons etc

from IPython.display import clear_output
import ipywidgets as wdg
from ipywidgets import HBox
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import requests
import time

In [2]:
# this makes the plots show up in the notebook, copied from dashboard 
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [3]:
# this is the APIwrapper class, it handles talking to the API and makes sure i don't send too many requests

class APIwrapper:
    #API wrapper for accessing UKHSA data 
    _access_point = "https://api.ukhsa-dashboard.data.gov.uk"
    _last_access = 0.0

    def __init__(self, theme, sub_theme, topic, geography_type, geography, metric):
        # putting together the full URL path
        url_path = (f"/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/"
                   f"geography_types/{geography_type}/geographies/{geography}/metrics/{metric}")
        self._start_url = APIwrapper._access_point + url_path
        self._filters = None
        self._page_size = -1
        self.count = None

    def get_page(self, filters={}, page_size=5):
        #takes one page 
        if page_size > 365:
            raise ValueError("Max supported page size is 365")
        
        # start over if things changed
        if filters != self._filters or page_size != self._page_size:
            self._filters = filters
            self._page_size = page_size
            self._next_url = self._start_url
        
        if self._next_url == None:
            return []
        
        # rate limiting so it don't get blocked
        curr_time = time.time()
        deltat = curr_time - APIwrapper._last_access
        if deltat < 0.33:
            time.sleep(0.33 - deltat)
        APIwrapper._last_access = curr_time
        
        parameters = {x: y for x, y in filters.items() if y != None}
        parameters['page_size'] = page_size
        
        response = requests.get(self._next_url, params=parameters).json()
        self._next_url = response['next']
        self.count = response['count']
        
        return response['results']

    def get_all_pages(self, filters={}, page_size=365):
        #downloads all pages
        data = []
        while True:
            next_page = self.get_page(filters, page_size)
            if next_page == []:
                break
            data.extend(next_page)
        return data

In [4]:
# load the data from the JSON files
# these are global so I can update them later when refreshing

global icu_data
global positivity_data

with open("influenza_healthcare_ICUHDUadmissionRateByWeek.json", "rt") as INFILE:
    icu_data = json.load(INFILE)

with open("influenza_testing_positivityByWeek.json", "rt") as INFILE:
    positivity_data = json.load(INFILE)

In [5]:
# functions for wrangling the data

#turns string dates into pandas date objects
def parse_date(datestring):
    return pd.to_datetime(datestring, format="%Y-%m-%d")

# main data wrangling function: it creates a data frames and puts the data in
def wrangle_icu_data(rawdata):
    # get all the dates 
    dates = [entry['date'] for entry in rawdata]
    dates.sort()
    
    # work out the date range
    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])
    index = pd.date_range(startdate, enddate, freq='W-MON')
    
    # make an empty dataframe
    df = pd.DataFrame(index=index, columns=['ICU_Rate'])
    
    # fill it in with the actual data
    for entry in rawdata:
        date = parse_date(entry['date'])
        # some entries might not have data so need to check
        if entry['metric_value'] != None:
            value = float(entry['metric_value'])
        else:
            value = 0.0
        df.loc[date, 'ICU_Rate'] = value
    
    # fill in any gaps
    df.fillna(0.0, inplace=True)
    
   
    df = df.rename(columns={'ICU_Rate': 'ICU/HDU Admission Rate (per 100k)'})
    return df


def wrangle_positivity_data(rawdata):
    data_dict = {}
    age_groups = set()
    
    for entry in rawdata:
        date = entry['date']
        age = entry['age']
        value = entry['metric_value']
        
        # keep track of all the age groups
        age_groups.add(age)
        
        # put the data in the dictionary
        if date not in data_dict:
            data_dict[date] = {}
        data_dict[date][age] = value
    
    # sort everything
    dates = sorted(data_dict.keys())
    age_groups = sorted(age_groups)
    
    # make the date range
    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])
    index = pd.date_range(startdate, enddate, freq='W-MON')
    
    # create dataframe with age groups as columns
    df = pd.DataFrame(index=index, columns=age_groups)
    
    # fill it in
    for date_str, age_dict in data_dict.items():
        date = parse_date(date_str)
        for age, value in age_dict.items():
            # handle None values
            if value != None:
                val = float(value)
            else:
                val = 0.0
            df.loc[date, age] = val
    
    # fill gaps
    df.fillna(0.0, inplace=True)
    return df

In [6]:
# wrangle the data that was loaded
icu_df = wrangle_icu_data(icu_data)
positivity_df = wrangle_positivity_data(positivity_data)

# will cause a future warning error message

/tmp/ipykernel_394/2699445591.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0.0, inplace=True)
/tmp/ipykernel_394/2699445591.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0.0, inplace=True)


In [7]:
# function to download fresh data from the API - it calls the global variables from above and updates them using the functions I just defined

def access_api():
    structure = {
        "theme": "infectious_disease",
        "sub_theme": "respiratory",
        "topic": "Influenza",
        "geography_type": "Nation",
        "geography": "England"
    }
    
    # get ICU data
    structure["metric"] = "influenza_healthcare_ICUHDUadmissionRateByWeek"
    api = APIwrapper(**structure)
    icu_data = api.get_all_pages()
    
    # wait a bit between requests
    time.sleep(0.4)
    
    # get positivity data
    structure["metric"] = "influenza_testing_positivityByWeek"
    api = APIwrapper(**structure)
    positivity_data = api.get_all_pages()
    
    return icu_data, positivity_data

In [8]:
# button callback funtion 
def api_button_callback(button):

    global icu_df, positivity_df
    
    # using try-except so it doesn't crash if something goes wrong
    try:
        # change button to show it's working
        apibutton.description = 'Fetching...'
        apibutton.icon = 'spinner'
        apibutton.disabled = True
        
        # get new data
        new_icu_data, new_positivity_data = access_api()
        
        # wrangle 
        icu_df = wrangle_icu_data(new_icu_data)
        positivity_df = wrangle_positivity_data(new_positivity_data)
        
        # update button
        apibutton.description = 'Data Refreshed'
        apibutton.icon = 'check'
        apibutton.button_style = 'success'
        
        # refresh both graphs
        refresh_icu_graph()
        refresh_positivity_graph()
        
    except Exception as e:
        # if it fails just show an error
        apibutton.description = 'Update Failed'
        apibutton.icon = 'times'
        apibutton.button_style = 'danger'
        apibutton.disabled = False
        print("Error: " + str(e))
        print("Using old data.")


#creating the button
apibutton = wdg.Button(
    description='Refresh Data',
    disabled=False,
    button_style='info',
    tooltip='Click to download current data from UKHSA',
    icon='download'
)

# connect it to the callback
apibutton.on_click(api_button_callback)
        

---

## Data Refresh

Click the button to get the latest data from UKHSA. Both graphs will update automatically.

If the API is down or there's an error, you'll see a message and the old data will still be there.

In [9]:
display(apibutton)

Button(button_style='info', description='Refresh Data', icon='download', style=ButtonStyle(), tooltip='Click t…

---

## Graph 1: ICU/HDU Admission Rates Over Time

This time series graph shows the weekly rate of ICU/HDU admissions due to influenza in England. The rate is expressed per 100,000 population, allowing for comparison across different time periods.

Key observations:
- Seasonal patterns are clearly visible, with winter peaks
- ICU admissions typically lag behind community transmission
- The graph can be viewed on either a linear or logarithmic scale

**Controls:** Use the radio buttons below to switch between linear and logarithmic scales.

In [10]:
# graph 1: ICU admissions

#plots ICU data
def plot_icu_timeseries(scale):
    if scale == 'log':
        logscale = True
    else:
        logscale = False
    
    # making the plot
    ax = icu_df.plot(logy=logscale, figsize=(12, 6), color='darkred', linewidth=2)
    ax.set_xlabel('Date')
    ax.set_ylabel('Admission Rate (per 100,000 population)')
    ax.set_title('Influenza ICU/HDU Admission Rates in England')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show() # need this for updating to work properly


# widget for choosing scale
scale_widget = wdg.RadioButtons(
    options=['linear', 'log'],
    value='linear',
    description='Scale:',
    disabled=False
)


#updates the graph after getting new data
def refresh_icu_graph():
    current = scale_widget.value
    if current == 'linear':
        other = 'log'
    else:
        other = 'linear'
    scale_widget.value = other
    scale_widget.value = current


# connect widget to the plotting function
icu_graph = wdg.interactive_output(plot_icu_timeseries, {'scale': scale_widget})

In [11]:
display(scale_widget, icu_graph)

RadioButtons(description='Scale:', options=('linear', 'log'), value='linear')

Output()

---

## Graph 2: Testing Positivity by Age Group

This stacked area chart shows the percentage of influenza tests that return positive results, broken down by age groups. Each colored area represents a different age group, and the total height shows the combined positivity across all ages.

**Key observations:**
- Different age groups show varying susceptibility to influenza
- Young children (0-4 years) often show higher positivity rates
- Seasonal patterns affect all age groups but with different magnitudes

**Controls:** Use the dropdown menu below to select a specific year to examine.

In [12]:
# graph 2; positivity by age

#plots the positivity data for selected year 
def plot_positivity_by_age(year):
    # filter for the year
    year_data = positivity_df[positivity_df.index.year == year]
    
    # check there's actually data
    if len(year_data) == 0:
        print("No data for " + str(year))
        return
    
    # make stacked area chart
    ax = year_data.plot(kind='area', stacked=True, figsize=(12, 6), 
                        alpha=0.7, colormap='tab10')
    ax.set_xlabel('Date')
    ax.set_ylabel('Testing Positivity (%)')
    ax.set_title('Influenza Testing Positivity by Age Group - ' + str(year))
    ax.legend(title='Age Group', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# get list of years
available_years = sorted(positivity_df.index.year.unique())

# dropdown widget
year_widget = wdg.Dropdown(
    options=available_years,
    value=available_years[-1], # start with most recent
    description='Year:',
    disabled=False
)


def refresh_positivity_graph():
    #makes this graph update after new data 
    current = year_widget.value
    if len(available_years) > 1:
        # switch to different year temporarily
        if current != available_years[0]:
            other = available_years[0]
        else:
            other = available_years[1]
        year_widget.value = other
        year_widget.value = current


# connect widget to function
positivity_graph = wdg.interactive_output(plot_positivity_by_age, {'year': year_widget})

In [13]:
display(year_widget, positivity_graph)

Dropdown(description='Year:', index=8, options=(2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025), value=2…

Output()

---

## Data Source and Acknowledgments

**Data Source:** UK Health Security Agency (UKHSA)  
**API:** https://ukhsa-dashboard.data.gov.uk/  

**Metrics Used:**
- `influenza_healthcare_ICUHDUadmissionRateByWeek`: ICU/HDU admission rates per 100,000 population
- `influenza_testing_positivityByWeek`: Testing positivity percentages by age group

**License and Attribution:**  
Based on UK Government data published by the UK Health Security Agency and on the DIY Disease Tracking Dashboard Kit by Fabrizio Smeraldi (f.smeraldi@qmul.ac.uk). Released under the GNU GPLv3.0 or later.

**Author:** Hannah Thirsk  
**Student ID:** 250898710 
**Date:** December 4th 2025